In [1]:
import os, sys, time, random, csv
from pathlib import Path
from datetime import datetime

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from consultas import CONSULTAS

ARCHIVO = Path("../results/mediciones_wsl.csv")
NUCLEOS = ["1", "*"]          # local[1] y local[*]

def crear_sesion(nucleos):
    return (SparkSession.builder
            .master(f"local[{nucleos}]")
            .appName("medicion")
            .config("spark.driver.memory", "8g")
            .config("spark.python.worker.faulthandler.enabled", "true")
            .config("spark.sql.execution.pyspark.udf.faulthandler.enabled", "true")
            .config("spark.sql.optimizer.excludedRules",
                    "org.apache.spark.sql.catalyst.optimizer.InferFiltersFromConstraints")
            .getOrCreate())

def ejecutar(resultado):
    # Obliga a Spark a calcular el resultado completo, sin traerlo a la memoria del notebook
    if hasattr(resultado, "write"):
        resultado.write.format("noop").mode("overwrite").save()   # DataFrame, SQL y UDF
    else:
        resultado.foreach(lambda _: None)                         # RDD

def ya_medidas():
    if not ARCHIVO.exists():
        return set()
    with open(ARCHIVO, encoding="utf-8") as f:
        return {(r["nucleos"], r["ronda"], r["consulta"], r["variante"])
                for r in csv.DictReader(f) if r["estado"] == "OK"}

def medir(rondas):
    hechas = ya_medidas()
    nuevo = not ARCHIVO.exists()
    with open(ARCHIVO, "a", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        if nuevo:
            w.writerow(["fecha_hora", "nucleos", "ronda", "consulta", "variante", "segundos", "estado"])
        for nucleos in NUCLEOS:
            spark = crear_sesion(nucleos)
            df = spark.read.parquet("../data/secop.parquet")
            print(f"\n=== Sesión local[{nucleos}]: calentamiento ===")
            for v in ["rdd", "df", "sql", "udf"]:
                ejecutar(CONSULTAS["Q1"][v](df))
            for ronda in rondas:
                combos = [(q, v) for q in CONSULTAS for v in CONSULTAS[q]]
                random.Random(f"{nucleos}-{ronda}").shuffle(combos)
                for q, v in combos:
                    if (nucleos, str(ronda), q, v) in hechas:
                        continue
                    t0 = time.perf_counter()
                    try:
                        ejecutar(CONSULTAS[q][v](df))
                        estado = "OK"
                    except Exception as e:
                        estado = f"ERROR {type(e).__name__}"
                    seg = time.perf_counter() - t0
                    w.writerow([datetime.now().isoformat(timespec="seconds"),
                                nucleos, ronda, q, v, f"{seg:.3f}", estado])
                    f.flush()
                    print(f"local[{nucleos}] ronda {ronda}  {q} {v:>3}: {seg:7.1f} s  {estado}")
            spark.stop()
    print("\nListo.")

In [2]:
NUCLEOS = ["1"]
medir(rondas=[1, 2, 3])

26/10/28 07:31:11 WARN Utils: Your hostname, Ever resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/10/28 07:31:11 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/28 07:31:11 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable



=== Sesión local[1]: calentamiento ===


local[1] ronda 1  Q5 udf:     4.5 s  OK


local[1] ronda 1  Q1  df:     1.2 s  OK


local[1] ronda 1  Q2  df:     1.3 s  OK


...

local[1] ronda 3  Q5 sql:     1.3 s  OK


local[1] ronda 3  Q3 rdd:    12.9 s  OK


local[1] ronda 3  Q4 udf:     5.3 s  OK


local[1] ronda 3  Q4  df:     4.3 s  OK


local[1] ronda 3  Q3 udf:     3.3 s  OK

Listo.


In [3]:
NUCLEOS = ["*"]
medir(rondas=[1, 2, 3])

26/10/28 07:25:58 WARN Utils: Your hostname, Ever resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/10/28 07:25:58 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/28 07:25:59 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
                                                                                


=== Sesión local[*]: calentamiento ===


local[*] ronda 1  Q4 sql:     3.9 s  OK


local[*] ronda 1  Q1 udf:     1.3 s  OK


local[*] ronda 1  Q6 rdd:     4.5 s  OK


...

local[*] ronda 3  Q5 rdd:     4.9 s  OK
local[*] ronda 3  Q2 sql:     0.3 s  OK


local[*] ronda 3  Q3 udf:     1.4 s  OK


local[*] ronda 3  Q6 rdd:     3.7 s  OK
local[*] ronda 3  Q5  df:     0.7 s  OK


local[*] ronda 3  Q4 rdd:     6.4 s  OK
local[*] ronda 3  Q2 udf:     0.5 s  OK


local[*] ronda 3  Q1 udf:     1.1 s  OK
local[*] ronda 3  Q1 sql:     0.5 s  OK

Listo.


In [2]:
import platform, psutil

# 1. Guarda el piloto de Windows con su propio nombre
viejo = Path("../results/mediciones.csv")
if viejo.exists():
    viejo.rename("../results/mediciones_windows.csv")
    print("Piloto de Windows guardado como results/mediciones_windows.csv")

# 2. Registra el equipo en Linux
spark = crear_sesion("*")
info = f"""Sistema operativo: {platform.platform()}
Núcleos lógicos visibles: {os.cpu_count()}
Memoria RAM visible: {psutil.virtual_memory().total / 1e9:.1f} GB
Python: {sys.version.split()[0]}
Java: {spark.sparkContext._jvm.System.getProperty("java.version")}
Spark: {spark.version}"""
spark.stop()

print(info)
with open("../results/equipo_wsl.txt", "w", encoding="utf-8") as f:
    f.write(info)

Piloto de Windows guardado como results/mediciones_windows.csv


26/10/28 06:53:46 WARN Utils: Your hostname, Ever resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/10/28 06:53:46 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/28 06:53:47 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Sistema operativo: Linux-6.18.54.1-microsoft-standard-WSL2-x86_64-with-glibc2.39
Núcleos lógicos visibles: 16
Memoria RAM visible: 12.5 GB
Python: 3.11.9
Java: 17.0.20.1
Spark: 3.5.9


In [3]:
inicio = time.perf_counter()
medir(rondas=[0])
print(f"\nEl piloto tardó {(time.perf_counter() - inicio) / 60:.1f} minutos")


=== Sesión local[1]: calentamiento ===


local[1] ronda 0  Q1  df:     1.4 s  OK


local[1] ronda 0  Q1 udf:     3.2 s  OK


local[1] ronda 0  Q3 sql:     2.5 s  OK


local[1] ronda 0  Q4 udf:     7.5 s  OK


...

26/10/28 06:57:52 ERROR DAGScheduler: Failed to update accumulator 1 (org.apache.spark.api.python.PythonAccumulatorV2) for task 0
java.net.SocketException: Broken pipe
	at java.base/sun.nio.ch.NioSocketImpl.implWrite(NioSocketImpl.java:425)
	at java.base/sun.nio.ch.NioSocketImpl.write(NioSocketImpl.java:445)
	at java.base/sun.nio.ch.NioSocketImpl$2.write(NioSocketImpl.java:831)
	at java.base/java.net.Socket$SocketOutputStream.write(Socket.java:1035)
	at java.base/java.io.BufferedOutputStream.flushBuffer(BufferedOutputStream.java:81)
	at java.base/java.io.BufferedOutputStream.flush(BufferedOutputStream.java:142)
	at java.base/java.io.DataOutputStream.flush(DataOutputStream.java:128)
	at org.apache.spark.api.python.PythonAccumulatorV2.merge(PythonRDD.scala:747)
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$updateAccumulators$1(DAGScheduler.scala:1720)
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$updateAccumulators$1$adapted(DAGScheduler.scala:1711)
	at scala.collection.mut

local[*] ronda 0  Q5 udf:     2.0 s  OK


local[*] ronda 0  Q6 sql:     1.5 s  OK


local[*] ronda 0  Q5 rdd:     4.8 s  OK


local[*] ronda 0  Q3 sql:     0.6 s  OK

Listo.

El piloto tardó 3.4 minutos


In [ ]:
import pandas as pd

m = pd.read_csv(ARCHIVO, dtype={"nucleos": str})
p = m[(m.ronda == 0) & (m.estado == "OK")]

print("Errores en el piloto:", (m[m.ronda == 0].estado != "OK").sum())
print("\nSegundos por combinación:")
print(p.pivot_table(index=["consulta", "variante"], columns="nucleos", values="segundos").round(1))

horas = 3 * p.segundos.sum() / 3600
print(f"\nEstimación para las rondas 1, 2 y 3: {horas:.1f} horas")
print("Regla: si pasa de 8 horas, se quita la Q4.")

In [ ]:
import psutil, threading

def es_python(p):
    return (p.info["name"] or "").lower().startswith("python")

class Monitor:
    """Toma una muestra del estado del equipo cada medio segundo mientras corre una consulta."""
    def __init__(self):
        self.muestras, self._parar = [], threading.Event()
        self.defender = [p for p in psutil.process_iter(["name"])
                         if (p.info["name"] or "").lower() == "msmpeng.exe"]
        self.acceso_defender = True
        for p in self.defender:
            try:
                p.cpu_percent(None)
            except psutil.Error:
                self.acceso_defender = False
        self.python_base = sum(1 for p in psutil.process_iter(["name"]) if es_python(p))

    def _tomar(self):
        psutil.cpu_percent(None)
        disco0, swap0 = psutil.disk_io_counters(), psutil.swap_memory().used
        while not self._parar.is_set():
            time.sleep(0.5)
            cpu_def = None
            if self.acceso_defender and self.defender:
                try:
                    cpu_def = sum(p.cpu_percent(None) for p in self.defender) / psutil.cpu_count()
                except psutil.Error:
                    self.acceso_defender = False
            self.muestras.append({
                "cpu": psutil.cpu_percent(None),
                "memoria": psutil.virtual_memory().percent,
                "python": sum(1 for p in psutil.process_iter(["name"]) if es_python(p)) - self.python_base,
                "defender": cpu_def,
            })
        disco1 = psutil.disk_io_counters()
        self.disco_mb = (disco1.read_bytes + disco1.write_bytes
                         - disco0.read_bytes - disco0.write_bytes) / 1e6
        self.swap_mb = (psutil.swap_memory().used - swap0) / 1e6

    def __enter__(self):
        self._hilo = threading.Thread(target=self._tomar, daemon=True)
        self._hilo.start()
        return self

    def __exit__(self, *args):
        self._parar.set()
        self._hilo.join()

    def resumen(self):
        m = self.muestras
        prom = lambda k: sum(x[k] for x in m) / len(m)
        maxi = lambda k: max(x[k] for x in m)
        defender = [x["defender"] for x in m if x["defender"] is not None]
        print(f"  CPU total:          promedio {prom('cpu'):.0f} %, máximo {maxi('cpu'):.0f} %")
        print(f"  Memoria RAM:        promedio {prom('memoria'):.0f} %, máximo {maxi('memoria'):.0f} %")
        print(f"  Memoria virtual:    {self.swap_mb:+,.0f} MB durante la consulta")
        print(f"  Disco leído+escrito:{self.disco_mb:,.0f} MB")
        print(f"  Procesos Python:    máximo {maxi('python')} a la vez (además del notebook)")
        if defender:
            print(f"  Antivirus:          promedio {sum(defender)/len(defender):.1f} %, máximo {max(defender):.1f} % de CPU")
        else:
            print("  Antivirus:          Windows no permite leer su consumo")

for nucleos in ["1", "*"]:
    spark = crear_sesion(nucleos)
    df = spark.read.parquet("../data/secop.parquet")
    print(f"local[{nucleos}]: particiones de lectura = {df.rdd.getNumPartitions()}")
    with Monitor() as mon:
        t0 = time.perf_counter()
        ejecutar(CONSULTAS["Q5"]["rdd"](df))
        seg = time.perf_counter() - t0
    print(f"local[{nucleos}]: Q5 rdd = {seg:.1f} s")
    mon.resumen()
    print()
    spark.stop()

In [ ]:
for nucleos in ["1", "*"]:
    spark = crear_sesion(nucleos)
    sc = spark.sparkContext
    print(f"local[{nucleos}]:")
    for intento in range(1, 4):
        with Monitor() as mon:
            t0 = time.perf_counter()
            sc.parallelize(range(16), 16).map(lambda x: x).count()
            seg = time.perf_counter() - t0
        maximo_python = max(x["python"] for x in mon.muestras) if mon.muestras else 0
        print(f"  intento {intento}: {seg:5.1f} s   procesos Python: {maximo_python}")
    spark.stop()
    print()